In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4090


In [2]:
import os
os.environ["BNB_CUDA_VERSION"] = "124" 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import logging
from datasets import load_dataset
from transformers import (
AutoTokenizer,
AutoModelForImageTextToText,
BitsAndBytesConfig,
DataCollatorForLanguageModeling,
EarlyStoppingCallback,
TrainerCallback
)

from peft import (
LoraConfig,
get_peft_model,
prepare_model_for_kbit_training,
TaskType
)

from trl import SFTTrainer, SFTConfig

MODEL_NAME = os.path.realpath("../Qwen3-VL-4B-It")
OUTPUT_DIR = os.path.realpath("../outputs/qwen3_examassist_4b_domain_lora")

/opt/miniconda3/envs/qwen3_stage2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("finetune_examassist.log", encoding="utf-8"),
    ],
)
logger = logging.getLogger(__name__)

In [4]:
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
 
 
# ==============================================================================
# 1. Custom Data Collator — với debug counter để phát hiện template miss
# ==============================================================================
class CompletionOnlyCollator(DataCollatorForLanguageModeling):
    """
    Mask toàn bộ phần prompt (user), chỉ tính loss trên phần assistant.
    Thêm counter để phát hiện samples bị mask hoàn toàn do không tìm thấy template.
    """
 
    def __init__(self, response_template: str, tokenizer, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, **kwargs)
        self.response_template    = response_template
        self.response_token_ids   = tokenizer.encode(response_template, add_special_tokens=False)
        self._total_calls         = 0
        self._total_miss          = 0
 
    def torch_call(self, examples):
        batch  = super().torch_call(examples)
        labels = batch["labels"].clone()
        batch_miss = 0
 
        for i in range(len(examples)):
            input_ids = batch["input_ids"][i].tolist()
            tpl_ids   = self.response_token_ids
            tpl_len   = len(tpl_ids)
 
            response_start_idx = None
            # Tìm từ cuối lên để lấy lần xuất hiện CUỐI CÙNG (multi-turn an toàn hơn)
            for idx in range(len(input_ids) - tpl_len, -1, -1):
                if input_ids[idx : idx + tpl_len] == tpl_ids:
                    response_start_idx = idx + tpl_len
                    break
 
            if response_start_idx is not None:
                labels[i, :response_start_idx] = -100
            else:
                labels[i, :] = -100
                batch_miss += 1
 
        self._total_calls += len(examples)
        self._total_miss  += batch_miss
 
        if batch_miss > 0:
            logger.warning(
                f"[Collator] {batch_miss}/{len(examples)} samples trong batch này "
                f"không tìm thấy response template → bị mask hoàn toàn. "
                f"Tổng tích lũy: {self._total_miss}/{self._total_calls}"
            )
 
        batch["labels"] = labels
        return batch
 
 
# ==============================================================================
# 2. Callback: log val loss rõ ràng sau mỗi eval để phát hiện frozen sớm
# ==============================================================================
class EvalLossMonitorCallback(TrainerCallback):
    """
    In eval_loss ra rõ ràng sau mỗi evaluation step.
    Cảnh báo nếu val loss không thay đổi sau 2 lần eval liên tiếp.
    """
 
    def __init__(self):
        self._prev_eval_loss = None
        self._frozen_count   = 0
 
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        current_loss = metrics.get("eval_loss")
        if current_loss is None:
            return
 
        logger.info(f"[EvalMonitor] Step {state.global_step} — eval_loss = {current_loss:.6f}")
 
        if self._prev_eval_loss is not None:
            diff = abs(current_loss - self._prev_eval_loss)
            if diff < 1e-5:
                self._frozen_count += 1
                logger.warning(
                    f"[EvalMonitor] Val loss thay đổi rất ít (Δ={diff:.2e}). "
                    f"Frozen count: {self._frozen_count}. "
                    f"Kiểm tra collator template miss hoặc val set quá nhỏ."
                )
            else:
                self._frozen_count = 0
 
        self._prev_eval_loss = current_loss

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [6]:
dataset = load_dataset(
    "json",
    data_files={
    "train": "../data/domain/train-final.jsonl",
    "validation": "../data/domain/valid-final.jsonl"
    }
)

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")

# Validate cấu trúc mẫu đầu tiên
sample = dataset["train"][0]
assert "messages" in sample, "Dataset phải có trường 'messages'"
logger.info(f"Sample keys: {list(sample.keys())}")

# def formatting_func(examples):
#     texts = []
#     # Duyệt qua danh sách các "messages" trong batch dữ liệu
#     for messages in examples["messages"]:
#         text = tokenizer.apply_chat_template(
#             messages, 
#             tokenize=False, 
#             add_generation_prompt=False
#         )
#         texts.append(text)
#     return texts
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

2026-06-24 17:20:37,952 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
2026-06-24 17:20:37,961 [INFO] Sample keys: ['messages', 'source']


Train samples: 1880
Validation samples: 152


In [7]:

collator = CompletionOnlyCollator(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, 
)

In [8]:
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",   
    local_files_only=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False
model.enable_input_require_grads()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
2026-06-24 17:20:38,014 [WARNING] WARNING: BNB_CUDA_VERSION=124 environment variable detected; loading libbitsandbytes_cuda124.so.
This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=

Loading weights: 100%|██████████| 713/713 [00:00<00:00, 781.05it/s] 


In [9]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

In [10]:
# model = get_peft_model(model, peft_config) 
# model.print_trainable_parameters()

In [11]:
trainable_before = sum(1 for n, p in model.named_parameters() if p.requires_grad)
print(f"[TRƯỚC SFTTrainer] Số params trainable: {trainable_before}")

[TRƯỚC SFTTrainer] Số params trainable: 0


# Training Arguments

In [16]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,

    # Learning rate
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Logging
    eval_strategy="steps",           
    eval_steps=25,                   
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,              # tăng lên 3 để giữ thêm checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Precision
    bf16=True,
    fp16=False,

    # Optimizer
    optim="adamw_torch_fused",

    # Stability
    gradient_checkpointing=True,
    max_grad_norm=0.3,

    # Sequence length
    max_length=2048,
    packing=False,

    # Misc
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,

    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],

    args=training_args,

    processing_class=tokenizer,
    data_collator=collator,
    formatting_func=formatting_func,
    peft_config=peft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001),
               EvalLossMonitorCallback(),],
    
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/opt/miniconda3/envs/qwen3_stage2/lib/python3.11/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/opt/miniconda3/envs/qwen3_stage2/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Truncating eval dataset: 100%|██████████| 152/152 [00:00<00:00, 89353.08 examples/s]


In [13]:
import trl
import transformers
import peft

print(trl.__version__)
print(transformers.__version__)
print(peft.__version__)

0.21.0
5.9.0
0.15.2


# ====TRAINING====

In [14]:
# Xác minh LoRA params có requires_grad=True không
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print(f"Số params trainable: {len(trainable)}")
print(f"Ví dụ: {trainable[:3]}")

# Xác minh optimizer thực sự cập nhật các params này
for n, p in model.named_parameters():
    if p.requires_grad:
        print(f"{n}: norm={p.data.norm().item():.6f}")
        break

Số params trainable: 504
Ví dụ: ['model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight', 'model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight', 'model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight']
model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight: norm=1.641309


In [17]:
trainer.train()

trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)

Step,Training Loss,Validation Loss
25,1.953504,0.554912
50,1.083453,0.447107
75,0.966530,0.426955
100,0.851029,0.425054
118,0.868014,0.424215


2026-06-24 17:26:34,938 [INFO] [EvalMonitor] Step 25 — eval_loss = 0.554912
2026-06-24 17:30:04,970 [INFO] [EvalMonitor] Step 50 — eval_loss = 0.447107
2026-06-24 17:33:33,779 [INFO] [EvalMonitor] Step 75 — eval_loss = 0.426955
2026-06-24 17:36:57,253 [INFO] [EvalMonitor] Step 100 — eval_loss = 0.425054
2026-06-24 17:39:10,784 [INFO] [EvalMonitor] Step 118 — eval_loss = 0.424215


('/home/drnguyenvinh/Exam-Assistant/Training_Model/outputs/qwen3_examassist_4b_domain_lora/tokenizer_config.json',
 '/home/drnguyenvinh/Exam-Assistant/Training_Model/outputs/qwen3_examassist_4b_domain_lora/chat_template.jinja',
 '/home/drnguyenvinh/Exam-Assistant/Training_Model/outputs/qwen3_examassist_4b_domain_lora/tokenizer.json')

In [ ]:
sample = dataset["train"][0]

text = tokenizer.apply_chat_template(
    sample["messages"],
    tokenize=False,
    add_generation_prompt=False
)

input_ids = tokenizer(
    text,
    add_special_tokens=False
)["input_ids"]

print("="*80)
print(text)
print("="*80)

print("\nTemplate:")
print(repr(RESPONSE_TEMPLATE))

tpl_ids = tokenizer.encode(
    RESPONSE_TEMPLATE,
    add_special_tokens=False
)

print("\nTemplate token ids:")
print(tpl_ids)

print("\nSearching...")

found = False

for idx in range(len(input_ids)-len(tpl_ids)+1):
    if input_ids[idx:idx+len(tpl_ids)] == tpl_ids:
        print("FOUND at", idx)
        found = True
        break

if not found:
    print("NOT FOUND")

<|im_start|>system
Bạn là trợ lý AI hỗ trợ giám thị coi thi. Chỉ trả lời dựa trên ngữ cảnh được cung cấp. Trả lời ngắn gọn, chính xác và không suy diễn ngoài dữ liệu.<|im_end|>
<|im_start|>user
Context:
Quy trình nộp bài EOS từ FALL2025. SV giơ tay báo GT. GT lấy Submission Code (SC) và Response Code (RC) trên E360. GT nhập SC cho SV. SV tick I want to finish the exam rồi bấm Finish. Nếu hiện Exam finished successfully và RC khớp với RC trên E360 thì SV mới được ký tên trên E360.

Question:
GT hỏi: exam finished successfully nghĩa là gì<|im_end|>
<|im_start|>assistant
Nộp bài thành công.<|im_end|>


Template:
'<|im_start|>assistant\n'

Template token ids:
[151644, 77091, 198]

Searching...
FOUND at 152


In [ ]:
miss_count = 0

for i in range(len(dataset["train"])):

    sample = dataset["train"][i]

    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    input_ids = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    tpl_ids = tokenizer.encode(
        RESPONSE_TEMPLATE,
        add_special_tokens=False
    )

    found = False

    for idx in range(len(input_ids)-len(tpl_ids)+1):
        if input_ids[idx:idx+len(tpl_ids)] == tpl_ids:
            found = True
            break

    if not found:

        miss_count += 1

        print("\n"+"="*80)
        print("MISS SAMPLE", i)
        print("Length =", len(input_ids))
        print(text[-500:])
        print("="*80)

        if miss_count >= 5:
            break

print("\nTotal miss =", miss_count)


Total miss = 0


In [ ]:
import numpy as np

lengths = length_dataset["train"]["length"]

print(np.percentile(lengths,[50,90,95,99]))

[ 188.5  1111.   1303.   1322.21]
